In [ ]:
import pandas as pd
import warnings
import numpy as np
from sklearn.naive_bayes import GaussianNB
from rdkit import Chem
from tqdm import tqdm
import warnings
from data_processing import dataset_process
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# 定义文件路径
dataset_path = "./SAM_data_PCE_15%_0721.csv"
hl_path = "./SAM_data_PCE_15%_0721_with_HOMO_LUMO.csv"

In [ ]:
dataset = pd.read_csv(dataset_path)
dataset_homo_lumo = pd.read_csv(hl_path)

In [ ]:
# 指定所需的特征
features_to_include = ['maccs', 'estate', 'substrate', 'pvk_ions', 'pvk_bandgap'] # 'substrate', 'pvk_ions', 'pvk_bandgap'

# 调用 dataset_process 函数
processed_dataset = dataset_process(dataset_path, features_to_include)

In [ ]:
processed_dataset

In [ ]:
processed_dataset.iloc[:,1:-1]

In [ ]:
#按骨架核心分类
df_copy = processed_dataset.copy()

df_1 = pd.DataFrame(columns=df_copy.columns)
df_2 = pd.DataFrame(columns=df_copy.columns)
df_3 = pd.DataFrame(columns=df_copy.columns)
df_4 = pd.DataFrame(columns=df_copy.columns)
df_5 = pd.DataFrame(columns=df_copy.columns)
df_6 = pd.DataFrame(columns=df_copy.columns)
df_7 = pd.DataFrame(columns=df_copy.columns)

# 链状
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 0:
            df_1 = pd.concat([df_1, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 萘酰胺
patt_1 = Chem.MolFromSmiles('O=C1NC(=O)C2=CC=CC3=C2C1=CC=C3')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_1):
        df_2 = pd.concat([df_2, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 三苯胺
patt_2 = Chem.MolFromSmiles('C1=CC=C(C=C1)N(C1=CC=CC=C1)C1=CC=CC=C1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_2):
        df_3 = pd.concat([df_3, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 咔唑
patt_3 = Chem.MolFromSmiles('N1C2=C(C=CC=C2)C2=C1C=CC=C2')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_3):
        df_4 = pd.concat([df_4, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 联噻吩
patt_4 = Chem.MolFromSmiles('S1C=CC=C1C1=CC=CS1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_4):
        df_5 = pd.concat([df_5, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 单环
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 1:
            df_6 = pd.concat([df_6, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 其他
df_7 = df_copy.copy()

In [ ]:
print(df_1.shape)
print(df_2.shape)
print(df_3.shape)
print(df_4.shape)
print(df_5.shape)
print(df_6.shape)
print(df_7.shape)

In [ ]:
warnings.filterwarnings("ignore")

# 设置超参数空间
param_grid = {
              'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10]}


# 用于存储每组超参数组合的实验结果
results = []

# 遍历超参数空间
for var_smoothing in param_grid['var_smoothing']:
    #print(var_smoothing)
    # 用于存储当前超参数组合的实验结果
    param_results = []
            
    for i in tqdm(range(5), desc="Training Progress"):
        dfs = [df_1, df_2, df_3, df_4, df_5, df_6]
        df_test = pd.DataFrame()
        df_train = pd.DataFrame()
    
        for j, df_sub in enumerate(dfs, start=1):
            if len(df_sub) <= 10:
                test_indices = np.random.choice(df_sub.index, size=1, replace=False)
            elif 10 < len(df_sub) <= 20:
                test_indices = np.random.choice(df_sub.index, size=2, replace=False)
            elif 20 < len(df_sub) <= 30:
                test_indices = np.random.choice(df_sub.index, size=3, replace=False)
            elif 30 < len(df_sub) <= 40:
                test_indices = np.random.choice(df_sub.index, size=4, replace=False)
            elif 40 < len(df_sub) <= 50:
                test_indices = np.random.choice(df_sub.index, size=5, replace=False)
            elif 50 < len(df_sub):
                test_indices = np.random.choice(df_sub.index, size=6, replace=False)            
            else:
                test_indices = []
            # print(f"Test set indices for df_{j}: {test_indices}") 
                    
            # 将测试集数据添加到 df_test，保持原始索引号
            df_test = pd.concat([df_test, df_sub.loc[test_indices]], ignore_index=False)
    
            # 将剩余的数据添加到 df_train，保持原始索引号
            train_indices = df_sub.index.difference(test_indices)
            df_train = pd.concat([df_train, df_sub.loc[train_indices]], ignore_index=False)
            # 分离特征和标签
            df_train_X = df_train.iloc[:, 1:-1].astype(float)
            df_test_X = df_test.iloc[:, 1:-1].astype(float)
            df_train_label = df_train['label'].astype(int)
            df_test_label = df_test['label'].astype(int)
        # 训练模型
        model = GaussianNB(var_smoothing=var_smoothing)
        model.fit(df_train_X, df_train_label)
        # 计算精度
        accuracy = model.score(df_test_X, df_test_label)
        predicted_labels = model.predict(df_test_X)
        predicted_labels1 = model.predict(df_train_X)
        #print(predicted_labels)
        #print(predicted_labels1)
        # 记录实验结果
        param_results.append(accuracy)
        #print(param_results)
    # 计算平均值
    mean_accuracy = np.mean(param_results)
    std_deviation = np.std(param_results)
    # 记录当前超参数组合的实验结果
    results.append({
                'var_smoothing': var_smoothing,
                'mean_accuracy': mean_accuracy,
                'std_deviation': std_deviation
            })
df_results = pd.DataFrame(results)
df_results.to_csv("./1204/input_combination_feature/1204_SAM_data_PCE_15%_with_maccs_estate_NB.csv")